# 11주차 ③ 인코더 블록 조립과 어텐션 맵 — 실습 6~8

**목표**: 잔차 연결 + LayerNorm + FFN 을 붙여 **인코더 블록을 완성**하고,
마스킹 전후 어텐션 행렬을 비교하며, **어텐션 맵을 읽는 법**을 익힌다.

> **과제 제출 대상 노트북입니다.** 마지막에 `transformer_blocks.py` 로 정리합니다.

### 잔차 연결 — 7주차에 이미 본 것 ★

```
   7주차 ResNet :   out = x + F(x)          층이 깊어져도 기울기가 살아 있다
   11주차       :   x = x + Attention(x)    ★ 완전히 같은 아이디어

     이유도 같다 :  ① 기울기가 덧셈을 통해 그대로 앞으로 흐른다
                    ② 블록이 "아무것도 안 하기"를 쉽게 배울 수 있다 (F(x)=0)
                    ③ 그래서 12층·24층·96층을 쌓을 수 있다
```

### LayerNorm — BatchNorm 과 무엇이 다른가 ★★

```
   (B, T, C) 텐서에서 "무엇의 평균을 빼는가"

            토큰1  [ ■ ■ ■ ■ ■ ■ ■ ■ ]  ← LayerNorm 은 이 한 줄 안에서 평균·분산
            토큰2  [ ■ ■ ■ ■ ■ ■ ■ ■ ]
            토큰3  [ ■ ■ ■ ■ ■ ■ ■ ■ ]
                     ↑
                  BatchNorm 은 이 한 칸을 세로로(배치 전체)
```

| | BatchNorm | LayerNorm |
|---|---|---|
| 정규화 축 | 배치 | **특징(C)** |
| 배치 크기 의존 | **있다** (작으면 불안정) | 없다 |
| 문장 길이가 제각각일 때 | 패딩이 통계를 오염시킨다 | **영향 없다** ★ |
| 추론 시 이동평균 필요 | 필요 | **불필요** |

> **핵심 메시지 ★★ (기말 출제 지점)**: 시퀀스에 **LayerNorm 을 쓰는 이유**는
> ① 문장 길이가 제각각이라 **배치 통계가 신뢰할 수 없고**,
> ② 배치 크기에 **의존하지 않아** 추론에서 안전하기 때문입니다.

In [ ]:
# 셀 1 — 정말 토큰 하나 안에서 정규화되나
import torch, torch.nn as nn, math
torch.manual_seed(42)

x  = torch.randn(2, 3, 8) * 5 + 10          # (B=2, T=3, C=8)
ln = nn.LayerNorm(8)                         # ★ 마지막 축(C)에 대해
y  = ln(x)
print("정규화 전 토큰[0,0] 평균/표준편차 :",
      round(x[0,0].mean().item(), 3), round(x[0,0].std(unbiased=False).item(), 3))
print("정규화 후 토큰[0,0] 평균/표준편차 :",
      round(y[0,0].mean().item(), 3), round(y[0,0].std(unbiased=False).item(), 3),
      " ← 0, 1 ★")

# BatchNorm 과 대비 : 정규화되는 축이 다르다
bn = nn.BatchNorm1d(3)                       # (B, T, C) 에서 T 축을 채널로 본다
print("\nLayerNorm  : 한 토큰의 C 차원 안에서 (가로)")
print("BatchNorm  : 같은 채널을 배치에 걸쳐  (세로)")

### 피드포워드 네트워크(FFN)

```
   어텐션은 "토큰끼리 정보를 섞는" 층이다.
   그런데 섞기만 하면 각 토큰 자체를 가공하는 단계가 없다.

     FFN :  각 토큰에 독립적으로 적용되는 2층 MLP  (5주차 그것)
        Linear(C, 4C)  →  GELU/ReLU  →  Linear(4C, C)
                          ↑
                     중간을 4배로 부풀린다 (관례)
```

> **핵심 메시지**: 블록의 역할 분담 — **어텐션은 "누구를 볼까"(토큰 간),
> FFN 은 "본 것을 어떻게 가공할까"(토큰 내)** 입니다.
> 블록 파라미터의 **약 2/3 가 FFN** 입니다.

## 실습 6 — 인코더 블록 조립 ★★

```
   Pre-LN 방식 (요즘 표준, 학습이 안정적)

     x ──┬──────────────────────────────┐
         │  LayerNorm → MultiHeadAttn   │
         └──────────────→ (+) ←─────────┘        x = x + Attn(LN(x))
                           │
         ┌─────────────────┴────────────┐
         │  LayerNorm → FFN             │
         └──────────────→ (+) ←─────────┘        x = x + FFN(LN(x))
                           │
                           ▼   출력 shape = 입력 shape (B, T, C)  ★
```

> **핵심 메시지 ★ (출제 지점)**: 인코더 블록의 **입력과 출력 shape 이 같습니다.**
> 그래서 **그냥 쌓을 수 있습니다.** BERT-base 는 이 블록이 **12개**입니다.

In [ ]:
# 셀 2 — 2교시 부품을 가져온다
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    alpha = torch.softmax(scores, dim=-1)
    return alpha @ V, alpha

class MultiHeadAttention(nn.Module):
    def __init__(self, C, H):
        super().__init__()
        assert C % H == 0
        self.H, self.d = H, C // H
        self.W_q = nn.Linear(C, C, bias=False)
        self.W_k = nn.Linear(C, C, bias=False)
        self.W_v = nn.Linear(C, C, bias=False)
        self.W_o = nn.Linear(C, C)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.H, self.d).transpose(1, 2)   # (B,H,T,d)
        k = self.W_k(x).view(B, T, self.H, self.d).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.H, self.d).transpose(1, 2)
        out, alpha = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).contiguous().view(B, T, C)         # (B,T,C)
        return self.W_o(out), alpha

print("부품 준비 완료")

In [ ]:
# 셀 3 — 인코더 블록 ★★ 오늘의 결과물
class EncoderBlock(nn.Module):
    def __init__(self, C, H, ff_mult=4, p_drop=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(C)
        self.attn = MultiHeadAttention(C, H)
        self.ln2  = nn.LayerNorm(C)
        self.ffn  = nn.Sequential(
            nn.Linear(C, ff_mult * C),      # (B,T,C) → (B,T,4C)
            nn.GELU(),
            nn.Linear(ff_mult * C, C),      # (B,T,4C) → (B,T,C)
        )
        self.drop = nn.Dropout(p_drop)      # 6주차 과적합 대응 ★

    def forward(self, x, mask=None):
        a, alpha = self.attn(self.ln1(x), mask)
        x = x + self.drop(a)                # ★ 잔차 ① — 7주차 ResNet 과 같다
        x = x + self.drop(self.ffn(self.ln2(x)))   # ★ 잔차 ②
        return x, alpha

blk = EncoderBlock(C=64, H=8)
x = torch.randn(2, 8, 64)                   # (B=2, T=8, C=64)
y, alpha = blk(x)
print("입력 :", x.shape, "→ 출력 :", y.shape, " ← 같다 ★")
print("어텐션 :", alpha.shape)               # (2, 8, 8, 8) = (B, H, T, T)

# 파라미터의 어디가 큰가
n_attn = sum(p.numel() for p in blk.attn.parameters())
n_ffn  = sum(p.numel() for p in blk.ffn.parameters())
print(f"\n어텐션 {n_attn:,} 개 / FFN {n_ffn:,} 개  ← FFN 이 약 2/3 ★")

In [ ]:
# 셀 4 — 12층 = BERT-base 의 구조 ★
class Encoder(nn.Module):
    def __init__(self, C, H, n_layers):
        super().__init__()
        self.blocks = nn.ModuleList([EncoderBlock(C, H) for _ in range(n_layers)])

    def forward(self, x, mask=None):
        maps = []
        for b in self.blocks:
            x, a = b(x, mask)
            maps.append(a)                  # 층마다 어텐션 맵 보관
        return x, maps

enc = Encoder(C=64, H=8, n_layers=12)
y, maps = enc(torch.randn(2, 8, 64))
print("출력 :", y.shape, "| 어텐션 맵 :", len(maps), "장")

n = sum(p.numel() for p in enc.parameters())
print(f"\n파라미터(C=64)  : {n:,} 개")
print(f"BERT-base(C=768) : 약 110,000,000 개  ← 구조는 같고 크기만 다르다 ★")

> **핵심 메시지 ★★**: **여러분이 방금 BERT 의 구조를 짰습니다.**
> 다음 주에 쓸 BERT 는 여기서 `C=768, H=12, n_layers=12` 로 키우고
> **수억 문장으로 몇 주 학습시킨 것**입니다. **구조는 오늘 것과 같습니다.**

### 인코더·디코더와 마스킹 (개념)

```
   인코더 (BERT 계열)        문장 전체를 이해한다
        모든 토큰이 앞뒤를 다 본다              → 분류·검색·임베딩

   디코더 (GPT 계열)         다음 단어를 만든다
        t 번째 토큰은 t 이전만 볼 수 있다  ★    → 생성

     왜 가려야 하나 :  학습할 때 정답 문장이 통째로 들어간다.
                      가리지 않으면 "다음 단어"를 커닝하게 된다  ★
```

| 구조 | 대표 모델 | 오늘 만든 것과의 관계 |
|---|---|---|
| 인코더만 | BERT · ViT · **12주차** | **오늘 만든 것** ★ |
| 디코더만 | GPT · LLaMA | 오늘 것 + **인과 마스킹** |
| 인코더-디코더 | T5 · 번역 모델 | 둘을 잇고 **크로스 어텐션** 추가 |

```
   인과 마스크 (causal mask), T=4
              보는 대상 →
        나는  [ 1  0  0  0 ]      "나는" 은 자기만 본다
        어제  [ 1  1  0  0 ]      "어제" 는 앞 2개
      영화를  [ 1  1  1  0 ]
        봤다  [ 1  1  1  1 ]      마지막은 전부
```

## 실습 7 — 마스킹 전후 어텐션 비교

In [ ]:
# 셀 5 — 하삼각 마스크를 만든다
T = 4
causal = torch.tril(torch.ones(T, T))        # (T, T) 하삼각
print(causal)

x = torch.randn(1, T, 64)
mha = MultiHeadAttention(C=64, H=4)

_, a_free   = mha(x)                          # 마스킹 없음
_, a_masked = mha(x, mask=causal)             # 마스킹 있음

print("\n[마스킹 없음] 헤드 0")
print(a_free[0, 0].detach().numpy().round(3))
print("\n[인과 마스킹] 헤드 0")
print(a_masked[0, 0].detach().numpy().round(3), " ← 우상단이 전부 0 ★")
print("\n각 행의 합은 여전히 1 :", a_masked[0, 0].sum(-1).detach().numpy().round(4))

> **관찰 포인트 ★★**: 가린 자리는 **정확히 0** 이고, 남은 자리들의 **합은 여전히 1** 입니다.
> `-inf` → `e^(-inf) = 0` → softmax 가 **남은 것들끼리 다시 비율을 맞춘** 결과입니다.
> **10주차 패딩 마스킹과 완전히 같은 기법**입니다.

In [ ]:
# 셀 6 — 패딩 마스크도 같은 방식
pad_mask = torch.tensor([[1., 1., 1., 0.]])          # 마지막 토큰이 <pad>
pad_mask = pad_mask.unsqueeze(1).unsqueeze(1)        # (B,1,1,T) ← 브로드캐스팅용
_, a_pad = mha(x, mask=pad_mask)
print("패딩 마스킹 — 마지막 열이 0 :")
print(a_pad[0, 0].detach().numpy().round(3))

# 실제 모델은 둘을 곱해서 함께 쓴다
both = causal * pad_mask                              # 브로드캐스팅으로 (1,1,T,T)
_, a_both = mha(x, mask=both)
print("\n인과 + 패딩 함께 :")
print(a_both[0, 0].detach().numpy().round(3))

> **핵심 메시지**: 마스크는 **`(T,T)` 인과형**과 **`(B,1,1,T)` 패딩형** 두 가지가 있고,
> 실제 모델은 **둘을 곱해서 함께** 씁니다. 원리는 하나 — *"가릴 자리를 `-inf` 로."*

## 실습 8 — 어텐션 맵 시각화 ★

> ⚠️ **학습을 안 한 모델이므로 무늬가 무작위입니다.**
> 여기서 보는 것은 **"어떻게 읽는가"** 이지 **"무엇이 보이는가"** 가 아닙니다.
> 의미 있는 무늬는 **다음 주 학습된 BERT 에서** 봅니다.

In [ ]:
# 셀 7 — 헤드별 어텐션 맵
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

tokens = ["나는", "어제", "친구와", "본", "영화가", "정말", "재미없었다", "."]
T, C, H = len(tokens), 64, 4

x = torch.randn(1, T, C)                      # (B=1, T=8, C=64)
mha = MultiHeadAttention(C=C, H=H)
_, alpha = mha(x)                             # (1, H, T, T)

fig, axes = plt.subplots(1, H, figsize=(4 * H, 4))
for h in range(H):
    ax = axes[h]
    ax.imshow(alpha[0, h].detach().numpy(), cmap="viridis", vmin=0)
    ax.set_xticks(range(T)); ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticks(range(T)); ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f"헤드 {h}")
axes[0].set_ylabel("Query (누가 보는가)")
fig.suptitle("어텐션 맵 — 어느 토큰이 어느 토큰을 봤는가")
plt.tight_layout(); plt.show()

In [ ]:
# 셀 8 — 읽는 법 연습
h, i = 0, 6                                    # 헤드 0, "재미없었다" 행
row = alpha[0, h, i].detach()
order = row.argsort(descending=True)[:3]
print(f'"{tokens[i]}" 가 가장 많이 본 토큰 3개 (헤드 {h}):')
for j in order:
    print(f"   {tokens[j]:8s} {row[j]:.3f}")

print("\n행 합 :", round(row.sum().item(), 4), " ← 1.0")

```
   어텐션 맵 읽는 법  ★

     세로축(행) = Query  "이 토큰이"
     가로축(열) = Key    "저 토큰을 이만큼 봤다"
     밝을수록 큰 값,  한 행의 합 = 1
```

> **핵심 메시지 ★**: 어텐션 맵은 **"어디를 봤다"** 를 보여 주지만
> **"왜 그렇게 판단했다"** 는 아닙니다. **9주차 Grad-CAM 과 똑같은 한계**입니다.

In [ ]:
# 셀 9 — 과제 소재 : H=1 과 H=4 를 비교한다 ★
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

torch.manual_seed(0)
mha1 = MultiHeadAttention(C=C, H=1)
_, a1 = mha1(x)
axes[0].imshow(a1[0, 0].detach().numpy(), cmap="viridis", vmin=0)
axes[0].set_title("H=1 (헤드 하나)"); axes[0].set_xticks([]); axes[0].set_yticks([])

torch.manual_seed(0)
mha4 = MultiHeadAttention(C=C, H=4)
_, a4 = mha4(x)
for h in range(4):
    axes[h+1].imshow(a4[0, h].detach().numpy(), cmap="viridis", vmin=0)
    axes[h+1].set_title(f"H=4 · 헤드 {h}"); axes[h+1].set_xticks([]); axes[h+1].set_yticks([])

fig.suptitle("헤드 수를 바꾸면 — 무늬가 하나 vs 넷")
plt.tight_layout(); plt.show()

print("H=1 : 무늬가 한 장뿐이다 → 한 가지 관계만 볼 수 있다")
print("H=4 : 네 장이 서로 다르다 → 여러 관계를 동시에 볼 수 있다  ★")

> **과제 소재 ★**: 위 그림이 *"헤드를 왜 나누는가"* 의 **시각적 답**입니다.
> 과제에 이 비교 그림 1장을 넣으세요.

## 코드를 파일로 정리 (필수 ★)

노트북 셀에 흩어진 클래스를 **`transformer_blocks.py`** 한 파일로 옮겨 적으세요.

```
   transformer_blocks.py 에 들어갈 것
     scaled_dot_product_attention
     MultiHeadAttention
     EncoderBlock
     Encoder
     positional_encoding        ← 2교시 셀 8 에서 가져온다
```

> **왜 파일로 만드나**: 노트북 셀에 흩어져 있으면 **"내가 트랜스포머를 짰다"** 가 안 남습니다.
> 파일로 묶으면 **미니 프로젝트에서 그대로 재사용**할 수 있고, 과제 제출물이기도 합니다.

In [ ]:
# 셀 10 — 파일에서 불러 써지는지 확인 (파일을 만든 뒤에 실행)
from transformer_blocks import EncoderBlock, positional_encoding

blk = EncoderBlock(C=64, H=8)
out, _ = blk(torch.randn(1, 8, 64) + positional_encoding(8, 64).unsqueeze(0))
print("불러오기 성공 :", out.shape)

---

### 과제 (마감 11/19 목 23:59)

| 항목 | 내용 |
|------|------|
| 제출물 | `22_encoder_block.ipynb`(출력 저장) / **`transformer_blocks.py`** / 어텐션 맵 1장 / **H=1 vs H=4 어텐션 맵 비교** / *"√d_k 로 나누는 이유"* 3줄 |
| 배점 | 과제 15점 중 **0.5점** |

> **채점 기준**: **shape 오류에는 관대하지만 설명은 엄격히** 봅니다.
> 코드가 조금 틀려도 **왜 √d_k 로 나누는지, 왜 헤드를 나누는지**
> 자기 말로 쓴 사람이 트랜스포머를 이해한 것입니다.

### 12주차 예고 ★★

> *"오늘 만든 블록을 12개 쌓고 위키피디아·뉴스 수억 문장으로 몇 주 학습시켜 놓은 것이
> **BERT** 입니다. 다음 주에는 그 BERT 를 내려받아 **한국어 영화 리뷰 감성 분류**에 맞춥니다.
> 10주차에 LSTM 으로 했던 그 데이터입니다 — **정확도가 얼마나 차이 나는지** 직접 비교합니다."*
>
> **준비**: 10주차 LSTM 정확도를 **찾아 둘 것**. `transformers`·`datasets` 설치 확인.

### 이 노트북 체크리스트

- [ ] 잔차 연결이 **7주차 ResNet 과 같은 것**임을 안다 ★
- [ ] LayerNorm 과 BatchNorm 의 **정규화 축 차이**를 말할 수 있다 ★★
- [ ] 시퀀스에 LayerNorm 을 쓰는 이유 2가지를 안다
- [ ] FFN 의 역할(토큰 내 가공)과 어텐션의 역할(토큰 간 혼합)을 구분한다
- [ ] 인코더 블록을 구현했고 **입출력 shape 이 같은 것**을 확인했다 ★
- [ ] 블록을 12층 쌓아 봤다
- [ ] 마스킹 후 어텐션이 **하삼각이 되고 행 합이 1** 인 것을 봤다 ★
- [ ] 어텐션 맵을 그리고 **행 = Query, 열 = Key** 로 읽을 수 있다 ★
- [ ] **코드를 `transformer_blocks.py` 로 정리했다** ★